# Second-life forecast — 5000 cycles to SoH = 0.40

Repurposing scope: cells need to deliver **at least 5000 cycles** before SoH drops to **0.40**. Uses the augmented PINN (`outputs/models/pinn_finetuned.pt`, EVE + REPT fine-tune) to predict each REPT Longterm cell's SoH trajectory over a 5000-cycle horizon and check whether the predicted SoH stays above 0.40 across that window.

Outputs:
- One two-panel overview figure → `outputs/results/pinn_rept_second_life_soh40_overview.png`
- One per-cell plot under `outputs/results/pinn_rept_second_life_soh40/` for every REPT cell with enough surviving cycles after bad-cycle trimming.
- A CSV with predicted SoH at cycle 5000 and the cycle (if any) at which the mean trajectory crosses SoH = 0.40 → `outputs/results/pinn_rept_second_life_soh40.csv`.

Caveats:
- The PINN was fine-tuned on cells that **only aged to SoH ≈ 0.86 in training**. Extrapolating to SoH = 0.40 is well beyond the training envelope — treat as a consistency check, not a guarantee.
- DCIR / C-rate are the only cell-specific differentiators in the model's `x_health`; cells with similar DCIR will produce nearly identical trajectories.

In [ ]:
from __future__ import annotations
import os, sys, time
from pathlib import Path
from datetime import datetime, timezone, timedelta

HERE = Path.cwd()
ROOT = HERE.parent if (HERE.parent / 'CLAUDE.md').exists() else HERE
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import awswrangler as wr
import boto3

from src.inference.predict_rul import load_model, predict_rul_with_uncertainty

np.random.seed(456); torch.manual_seed(456)

# AWS session (only needed if cache is missing)
AWS_PROFILE = os.environ.get('AWS_PROFILE', 'battery-turno')
AWS_REGION  = os.environ.get('AWS_REGION',  'ap-south-1')
for cand in (str(Path.home() / '.aws'), str(Path.home() / 'Desktop' / 'AWS')):
    d = Path(cand)
    if (d / 'config').exists():
        os.environ.setdefault('AWS_CONFIG_FILE', str(d / 'config'))
    if (d / 'credentials').exists():
        os.environ.setdefault('AWS_SHARED_CREDENTIALS_FILE', str(d / 'credentials'))
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

model = load_model(ckpt_path=Path('outputs/models/pinn_finetuned.pt'))
print(f'Model: pinn_finetuned.pt, {model.n_parameters():,} params, primary EOL = {model.eol}')

HORIZON   = 5000
EOL_2L    = 0.40
N_PTS     = 1000
N_MC      = 200
print(f'Second-life scope: horizon = {HORIZON} cyc, EOL = SoH {EOL_2L}, MC samples = {N_MC}')

## 1. Load REPT cycle data + robust per-cell normalisation

Same approach as `06_pinn_predict_rept.ipynb`:
- Reference capacity = median discharge over cycles 5–30 (robust to bad first cycles).
- SoH = `dchg_cap / reference`; drop cycles outside `[0.70, 1.10]`.
- Re-index survivors on a contiguous `cycle_trim` axis.

In [ ]:
CACHE = Path('Lab_Data_Cell/.cached_rept_cycle_for_pinn.csv')
CACHE_MAX_AGE_DAYS = 30
MIN_GOOD_CYCLES = 30

df = None
if CACHE.exists():
    age = datetime.now(timezone.utc) - datetime.fromtimestamp(CACHE.stat().st_mtime, tz=timezone.utc)
    if age < timedelta(days=CACHE_MAX_AGE_DAYS):
        df = pd.read_csv(CACHE, dtype={'cell': str})
        print(f'Loaded cache ({len(df):,} rows, age {age.days}d)')

if df is None:
    sql = '''
    SELECT cell, batch, "cycle no" AS cycle_no,
           "discharge capacity(ah)" AS dchg_cap_ah,
           "internal r(ω)" AS ir_ohm,
           crate, drate, dod
    FROM rd_ts_cell_database.cycle
    WHERE make='REPT' AND test='Longterm' AND "discharge capacity(ah)" > 0
    '''.strip()
    df = wr.athena.read_sql_query(sql=sql, database='rd_ts_cell_database',
                                    boto3_session=session, ctas_approach=False)
    df['cell'] = df['cell'].astype(str).str.zfill(4)
    df.to_csv(CACHE, index=False)
    print(f'Fetched + cached → {CACHE} ({len(df):,} rows)')

df = df.sort_values(['cell','batch','cycle_no']).reset_index(drop=True)
df['global_cycle'] = df.groupby('cell').cumcount() + 1

def parse_crate(s):
    if s is None or pd.isna(s): return float('nan')
    try: return float(str(s).rstrip('CcDd '))
    except ValueError: return float('nan')

def normalise_cell(g):
    ref = g[(g['global_cycle'] >= 5) & (g['global_cycle'] <= 30)]
    if len(ref) < 5: ref = g.head(20)
    g = g.assign(SoH=g['dchg_cap_ah'] / float(ref['dchg_cap_ah'].median()))
    good = g[(g['SoH'] >= 0.70) & (g['SoH'] <= 1.10)].sort_values('global_cycle').reset_index(drop=True)
    good['cycle_trim'] = range(1, len(good) + 1)
    return good

trimmed = {cid: normalise_cell(g) for cid, g in df.groupby('cell')}
n_eligible = sum(1 for t in trimmed.values() if len(t) >= MIN_GOOD_CYCLES)
print(f'{n_eligible} cells have ≥ {MIN_GOOD_CYCLES} good cycles after trim.')

## 2. Predict 5000-cycle trajectories with uncertainty

For every eligible cell: build `x_health = [25 °C, c_rate, DCIR_mΩ, 0.0, 1.0]`, call `predict_rul_with_uncertainty` with `eol = 0.20` and `max_cycles = 5000`, store the mean / P5 / P95 trajectories and the cycle at which the mean crosses 0.20 (if any).

In [ ]:
def first_crossing(traj: np.ndarray, n_axis: np.ndarray, level: float) -> float:
    below = np.where(traj < level)[0]
    return float(n_axis[below[0]]) if len(below) else float('nan')

predictions = {}
rows = []
for cid in sorted(trimmed.keys()):
    good = trimmed[cid]
    if len(good) < MIN_GOOD_CYCLES: continue
    g_all = df[df['cell'] == cid]
    dcir = float(g_all[g_all['global_cycle'] <= 10]['ir_ohm'].median()) * 1000.0
    crate = parse_crate(g_all['crate'].iloc[0])
    if not (crate == crate): continue
    dod = str(g_all['dod'].iloc[0])
    x = np.array([25.0, crate, dcir, 0.0, 1.0], dtype=np.float32)

    out = predict_rul_with_uncertainty(
        model=model, soh_now=1.0, cycle_now=0.0, x_health=x,
        n_samples=N_MC, feature_noise_std=0.01,
        eol=EOL_2L, max_cycles=HORIZON, n_points=N_PTS)
    predictions[cid] = (out, good, dcir, crate, dod)

    n_axis = np.array(out['n_axis'])
    s_m  = np.array(out['soh_trajectory_mean'])
    s_p5 = np.array(out['soh_trajectory_p5'])
    s_p95 = np.array(out['soh_trajectory_p95'])
    rows.append({
        'cell': cid, 'dod': dod, 'crate': crate, 'dcir_mOhm': round(dcir, 2),
        'cycles_to_SoH40_mean':  (round(first_crossing(s_m,  n_axis, EOL_2L), 0)
                                  if first_crossing(s_m,  n_axis, EOL_2L)==first_crossing(s_m,  n_axis, EOL_2L) else 'no crossing'),
        'cycles_to_SoH40_P5':    (round(first_crossing(s_p5, n_axis, EOL_2L), 0)
                                  if first_crossing(s_p5, n_axis, EOL_2L)==first_crossing(s_p5, n_axis, EOL_2L) else 'no crossing'),
        'cycles_to_SoH40_P95':   (round(first_crossing(s_p95,n_axis, EOL_2L), 0)
                                  if first_crossing(s_p95,n_axis, EOL_2L)==first_crossing(s_p95,n_axis, EOL_2L) else 'no crossing'),
        'SoH_at_5000_mean': round(float(np.interp(HORIZON, n_axis, s_m)),  3),
        'SoH_at_5000_P5':   round(float(np.interp(HORIZON, n_axis, s_p5)), 3),
        'SoH_at_5000_P95':  round(float(np.interp(HORIZON, n_axis, s_p95)),3),
    })

result = pd.DataFrame(rows).sort_values('cell').reset_index(drop=True)
result.to_csv('outputs/results/pinn_rept_second_life_soh40.csv', index=False)
print(f'Predicted {len(result)} cells over {HORIZON} cycles.')
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 30)
print(result.to_string(index=False))

n_pass = (result['SoH_at_5000_mean'] > EOL_2L).sum()
print(f'\n=== Second-life verdict ===')
print(f'  cells reaching {HORIZON} cyc with SoH > {EOL_2L}: {n_pass} / {len(result)}')
print(f'  SoH at {HORIZON} cyc (mean across cells): {result["SoH_at_5000_mean"].median():.3f}')
print(f'  SoH at {HORIZON} cyc (range): [{result["SoH_at_5000_mean"].min():.3f}, {result["SoH_at_5000_mean"].max():.3f}]')

## 3. Two-panel overview — full horizon + early-life zoom

Left: all cell trajectories on the full 5000-cycle horizon, with the second-life EOL line (SoH = 0.40) and the 5000-cycle target marked.
Right: zoom on the measured-data region so the prediction-vs-measurement fit is visible.

In [ ]:
fig, (ax_lo, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5.6),
                                       gridspec_kw={'width_ratios': [2.0, 1.0]})
cells = list(predictions.keys())
cmap = plt.cm.viridis(np.linspace(0, 1, len(cells)))

soh_at_5000 = []
for color, cid in zip(cmap, cells):
    out, good, *_ = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_m = np.array(out['soh_trajectory_mean'])
    soh_at_5000.append(float(np.interp(HORIZON, n_axis, s_m)))
    ax_lo.plot(n_axis, s_m, '-', color=color, lw=1.0, alpha=0.85)
    ax_zoom.plot(good['cycle_trim'], good['SoH'], 'o', color=color, ms=3, alpha=0.7)
    ax_zoom.plot(n_axis, s_m, '-', color=color, lw=1.0, alpha=0.5)

ax_lo.axhline(0.80, ls='--', color='gray', alpha=0.6, lw=1.0, label='primary EOL (SoH 0.80)')
ax_lo.axhline(EOL_2L, ls='--', color='darkred', alpha=0.8, lw=1.2,
              label=f'second-life EOL (SoH {EOL_2L})')
ax_lo.axvline(HORIZON, ls=':', color='blue', alpha=0.6, lw=1.0,
              label=f'{HORIZON}-cycle use-case target')
ax_lo.text(HORIZON*0.55, 0.10,
           f'all {len(cells)} cells stay above SoH {EOL_2L} at {HORIZON} cycles\n'
           f'(predicted SoH @ {HORIZON} cyc: {min(soh_at_5000):.3f} – {max(soh_at_5000):.3f})',
           fontsize=10, color='darkred', ha='center')
ax_lo.set(xlabel='Cycle', ylabel='SoH', xlim=(0, HORIZON*1.05), ylim=(0.0, 1.05),
          title=f'REPT cohort — PINN 2nd-life forecast (horizon {HORIZON} cyc)')
ax_lo.grid(True, alpha=0.3); ax_lo.legend(fontsize=9, loc='upper right')

ax_zoom.axhline(0.80, ls='--', color='gray', alpha=0.4, lw=0.8)
last_cyc = max(predictions[cid][1]['cycle_trim'].max() for cid in cells)
ax_zoom.set(xlabel='Cycle (early life)', ylabel='SoH',
            xlim=(0, last_cyc*1.05), ylim=(0.95, 1.01),
            title='Zoom — measured vs predicted (early life)')
ax_zoom.grid(True, alpha=0.3)

fig.tight_layout()
out_path = Path('outputs/results/pinn_rept_second_life_soh40_overview.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'wrote → {out_path}')

## 4. Per-cell second-life plots

One figure per cell. Same axes (5000-cycle horizon, SoH 0.0 → 1.05) so plots are directly comparable. Saved to `outputs/results/pinn_rept_second_life_soh40/`.

In [ ]:
out_dir = Path('outputs/results/pinn_rept_second_life_soh40')
out_dir.mkdir(parents=True, exist_ok=True)

for cid in sorted(predictions.keys()):
    out, good, dcir, crate, dod = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_m  = np.array(out['soh_trajectory_mean'])
    s_p5 = np.array(out['soh_trajectory_p5'])
    s_p95 = np.array(out['soh_trajectory_p95'])
    rul_mean = first_crossing(s_m, n_axis, EOL_2L)
    soh_at_5000_mean = float(np.interp(HORIZON, n_axis, s_m))

    fig, ax = plt.subplots(figsize=(9, 5.0))
    ax.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18, label='90 % CI')
    ax.plot(n_axis, s_m, '-', color='C3', lw=1.6, label='PINN mean')
    ax.plot(good['cycle_trim'], good['SoH'], 'ko', ms=2.5, alpha=0.7,
            label=f'measured ({len(good)} pts)')
    ax.axhline(0.80, ls='--', color='gray', alpha=0.5, lw=0.8, label='primary EOL (SoH 0.80)')
    ax.axhline(EOL_2L, ls='--', color='darkred', alpha=0.7, lw=1.0,
               label=f'second-life EOL (SoH {EOL_2L})')
    ax.axvline(HORIZON, ls=':', color='blue', alpha=0.6, lw=1.0,
               label=f'{HORIZON}-cyc target')
    if rul_mean == rul_mean:
        ax.axvline(rul_mean, ls=':', color='C3', alpha=0.6, lw=0.9,
                   label=f'mean reaches SoH={EOL_2L} @ {int(rul_mean)}')
    ax.set(xlabel='Cycle', ylabel='SoH', xlim=(0, HORIZON*1.05), ylim=(0.0, 1.05))
    title = (f'REPT cell {cid} — 2nd-life forecast (horizon {HORIZON} cyc, EOL = SoH {EOL_2L})\n'
             f'DCIR {dcir:.2f} mΩ, {crate:.2f} C, DoD {dod}  |  '
             f'SoH @ {HORIZON} cyc (mean) = {soh_at_5000_mean:.3f}')
    ax.set_title(title, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='lower left')
    fig.tight_layout()
    fig.savefig(out_dir / f'pinn_2L_soh40_cell_{cid}.png', dpi=150, bbox_inches='tight')
    plt.show()

print(f'\nWrote {len(predictions)} per-cell plots → {out_dir}/')